In [1]:

# #! echo $PWD
#! pip install ../utility_functions/
# #!pip install ~/fwiViz/utility_functions/

In [28]:
import fwiVis.fwiVis as fv
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
#import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain
from bs4 import BeautifulSoup # I mamba installed bs4
import requests

from datetime import date

In [65]:



# def make_df_of_fires(year = "2023", path_region = "Quebec_PostHoc", custom_path = "/home/jovyan/fireatlast_nrt/fireatlas/data/FEDSoutput-v3/"):

#     diroutdata = custom_path
#     spath = os.path.join(diroutdata, path_region, str(year), "Largefire")
#     fnms = [f for f in os.listdir(spath)]
#     print(fnms)
#     #fnms = fnms.sort()
#     tmp_ids = pd.DataFrame(fnms, columns=["ids"])
#     #tmp_ids = tmp_ids[~tmp_ids.ids.str.contains(".")]
#     print(tmp_ids)
#     tmp_ids = tmp_ids.ids.unique()
#     print(f'{len(tmp_ids)} unique ID found')

#     print("Reading in IDS")
#     ### reading in the ids
#     fires = pd.DataFrame()
#     for n,i in enumerate(tmp_ids, start = 0):
#         try:
#             foo = fv.load_large_fire(i, year = year, path_region= path_region, layer = "perimeter",  s3_path = False, custom_path = custom_path)
#             foo["fireID"] = str(i)
#         except Exception as e:
#             print("Error at ID: ",i)
#             print(e)
#             continue
#         ## Extract the period between 
#         fires = pd.concat([fires, foo])
#     return (fires)
#         #print(fires)
#             #fr_pd = pd.DataFrame(fires, columns=["lat", "lon", "farea", "data_source"])
#         #fires.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/"+"20_days_fire_stats_only_718270-99999_" +min_t + max_t + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")

In [66]:
# fires = make_df_of_fires()

In [29]:
## but wait, maybe Julia made a function for combined lf anyway


lf = gpd.read_file("/home/jovyan/fireatlast_nrt/fireatlas/data/FEDSoutput-v3/Quebec_V3/2023/CombinedLargefire/20230915PM/lf_perimeter.fgb")

In [30]:
len(lf.fireID.unique())

204

In [31]:
def get_largest_perimeter(df):
    df = df[(df.farea == df.farea.max()) & (df.t == df.t.max())]
    return(df)

In [32]:
new_lf = lf.groupby("fireID").apply(get_largest_perimeter)

/tmp/ipykernel_1598/3467081945.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  new_lf = lf.groupby("fireID").apply(get_largest_perimeter)


In [33]:
new_lf.explore()

In [43]:
new_lf = new_lf.to_crs("4326")
new_lf["lat_centroid"] = new_lf.geometry.centroid.y
new_lf["lon_centroid"] = new_lf.geometry.centroid.x

new_lf.columns

/tmp/ipykernel_1598/2034074402.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  new_lf["lat_centroid"] = new_lf.geometry.centroid.y
/tmp/ipykernel_1598/2034074402.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  new_lf["lon_centroid"] = new_lf.geometry.centroid.x


Index(['mergeid', 'ftype', 'n_pixels', 'n_newpixels', 'farea', 'fperim',
       'flinelen', 'duration', 'pixden', 'meanFRP', 't', 't_st', 't_ed',
       'fireID', 'isignition', 't_inactive', 'isactive', 'isdead',
       'mayreactivate', 'geom_counts', 'low_confidence_grouping', 'region',
       'primarykey', 'geometry', 'lat_centroid', 'lon_centroid'],
      dtype='object')

In [62]:
new_lf.fireID = new_lf.fireID.astype("int")
new_lf.mergeid = new_lf.mergeid.astype("int")

In [64]:
import datetime

now = datetime.datetime.now()
new_lf.to_csv("~/fwiVis/notebooks/data/Quebec_v3_final_perimeters_" + str(now.year)+str(now.month)+str(now.day)+".csv")